# SIVE — Harness minimale (senza agentsociety2)

Riscrittura di SIVE senza framework agentico generico, motivata dalla
valutazione di fattibilita' di agentsociety2 (vedi `agentsociety2_notes.tex`):
con quattro modelli diversi su quattro testati (Haiku, gpt-oss-20b/120b,
GPT-5.4 mini, Gemini 3.5 Flash) il ciclo generico di tool-decision del
framework ha generato tra 80 e 100 chiamate LLM per un singolo agente/
condizione, contro le 13 di DeepSeek per lo stesso compito -- un sintomo
ripetuto sull'unico meccanismo che SIVE non usa mai (non avendo ambiente
né strumenti reali da scegliere). Questo notebook implementa lo stesso
disegno sperimentale (sei condizioni iniziali, batteria a 5 item) con un
motore proprio: lista di messaggi per persona, chiamata diretta al
modello, nessun loop di decisione, nessun router/cache nascosti. Una
settima condizione (POSW2) viene aggiunta in un secondo momento, come
run incrementale (§14).

**Costo per agente/condizione, fisso indipendentemente dal modello**:
5 (PRE) + 3 (reazioni) + 5 (POST) = 13 chiamate.

Questo notebook si occupa esclusivamente della raccolta dati. Il
calcolo dei criteri di validazione C1–C7 è in `sive_temperature_analysis_v4.ipynb`;
il sotto-esperimento di caratterizzazione del rumore strumentale è in
`sive_c3_noise_subexperiment.ipynb`.


## 1 · Setup

Client OpenAI puro contro l'endpoint OpenRouter -- nessun framework,
nessun router LiteLLM, nessuna cache nascosta (lezione del 18 giugno
2026: `litellm.Router(cache_responses=True)` crea una cache *globale di
processo*, condivisa silenziosamente tra ogni Router creato nella stessa
sessione Colab). Qui non c'e' nulla da nascondere: ogni chiamata e'
esplicita nel codice sottostante.


In [1]:
!pip install openai -q

import os
import json
import re
import unicodedata
import difflib
import time
import numpy as np
import pandas as pd
from scipy import stats
from datetime import datetime as _dtclass
from pathlib import Path

from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("SIEVE")

from openai import AsyncOpenAI
client = AsyncOpenAI(
    api_key=userdata.get("SIEVE"),
    base_url="https://openrouter.ai/api/v1",
)

CURRENT_MODEL = "deepseek/deepseek-chat"  # cambia qui per testare un altro modello
CURRENT_TEMPERATURE = 0.9  # cambia qui per esperimenti di sweep sulla temperatura
print("Setup OK, modello attivo:", CURRENT_MODEL, "| temperatura:", CURRENT_TEMPERATURE)


Setup OK, modello attivo: deepseek/deepseek-chat | temperatura: 0.9


## 2 · Popolazione

Stesso file di sempre, `sive_120_completi.json` (120 profili, gruppi
LOW/MED/HIGH bilanciati 40/40/40). **Regola di disegno invariata**:
`gruppo`/`attitudini`/`T_latent` restano solo in `META`, lato Python --
non vengono mai esposti al modello (necessario per la validita' del
criterio C1).

**Novita' rispetto a prima**: dato che non passiamo piu' un dict
`profile` a un costruttore di framework, dobbiamo scrivere noi il testo
del system prompt che descrive la persona al modello -- prima questa
traduzione avveniva dentro agentsociety2, in modo opaco. Ora la
controlliamo (e possiamo iterarla) direttamente.


In [2]:
def build_profile(record):
    return {
        "name": record["name"],
        "age": record["age"],
        "gender": record["gender"],
        "education": record["education"],
        "occupation": record["occupation"],
        "marriage_status": record["marriage_status"],
        "personality": record["persona"],
        "bio": record["background_story"],
    }


def build_system_prompt(profile):
    return f"""Sei {profile['name']}, {profile['age']} anni, {profile['gender']}.
Titolo di studio: {profile['education']}. Occupazione: {profile['occupation']}.
Stato civile: {profile['marriage_status']}.
Personalita': {profile['personality']}.

La tua storia: {profile['bio']}

Vivi a Montelago. Rispondi sempre in italiano, restando fedele al tuo
carattere e al tuo background, parlando in prima persona. Non rivelare
mai di essere un modello linguistico o un personaggio simulato."""


with open("/content/sive_120_completi.json", encoding="utf-8") as f:
    PROFILI = json.load(f)
assert len(PROFILI) == 120 and [p["id"] for p in PROFILI] == list(range(1, 121))

META = {p["id"]: p for p in PROFILI}
GRUPPI = {g: [p["id"] for p in PROFILI if p["gruppo"] == g] for g in ["LOW", "MED", "HIGH"]}
print(f"OK {len(PROFILI)} profili | gruppi:", {g: len(v) for g, v in GRUPPI.items()})


OK 120 profili | gruppi: {'LOW': 40, 'MED': 40, 'HIGH': 40}


## 3 · Strumento di misura

Identico a v2 (agentsociety2), per mantenere i due harness confrontabili
sugli stessi identici stimoli/domande. `SCELTA`/`NUM` rinforzano
l'istruzione di formato direttamente nel testo della domanda.

**Domanda aperta (opzionale)**: `INCLUDE_OPEN_ENDED`, se attivato,
aggiunge un sesto item solo in fase POST -- testo libero, nessuna
normalizzazione, pensato per leggere "perche'" dietro casi anomali (es.
un'emozione fuori dall'insieme chiuso) senza dover indovinare a
posteriori.


In [3]:
EMOZIONI   = ['sollievo', 'preoccupazione', 'rabbia', 'speranza', 'indifferenza']
INTENZIONI = ['cercare più informazioni sulla rete idrica',
              'parlare con i vicini della situazione',
              'partecipare a incontri pubblici sul tema',
              'non cambierà nulla per me',
              'contattare il Comune per avere chiarimenti']

NUM    = ("IMPORTANTE: rispondi SOLO con un numero intero da 1 a 10, "
          "nessuna parola, nessuna spiegazione.")
SCELTA = ("IMPORTANTE: rispondi SOLO con una delle opzioni elencate, "
          "esattamente come scritta, senza aggiungere altro testo.")

ITEMS = [
    dict(name='fiducia_istituzione',
         prompt='Quanto ti fidi del Comune di Montelago nella gestione '
                'della rete idrica? '
                '(1=nessuna fiducia, 10=piena fiducia). ' + NUM,
         response_type='integer'),
    dict(name='credibilita',
         prompt='Quanto ritieni credibile la comunicazione del Comune '
                'sulla rete idrica? '
                '(1=per nulla credibile, 10=del tutto credibile). ' + NUM,
         response_type='integer'),
    dict(name='adeguatezza_info',
         prompt="Quanto ritieni adeguata l'informazione che hai ricevuto "
                'sulla rete idrica del tuo quartiere? '
                '(1=del tutto inadeguata, 10=molto adeguata). ' + NUM,
         response_type='integer'),
    dict(name='emozione',
         prompt="Quale emozione descrive meglio il tuo stato d'animo "
                "riguardo alla situazione della rete idrica nel tuo quartiere? "
                + SCELTA,
         response_type='choice', choices=EMOZIONI),
    dict(name='intenzione',
         prompt='Cosa pensi di fare nei prossimi giorni riguardo '
                'alla situazione della rete idrica? ' + SCELTA,
         response_type='choice', choices=INTENZIONI),
]

INCLUDE_OPEN_ENDED = True

OPEN_ENDED_ITEM = dict(
    name='motivazione_libera',
    prompt="In una o due frasi, perché ti senti così riguardo "
           "alla situazione della rete idrica?",
    response_type='open_text',
)

print("OK strumento SIVE v2 (5 item tematici + 1 aperto opzionale)")

OK strumento SIVE v2 (5 item tematici + 1 aperto opzionale)


## 4 · QC e normalizzazione

Identica a v2: match esatto -> contenimento (solo se un'unica opzione e'
contenuta) -> fuzzy match -> fallback, con log completo per il report
finale.


In [4]:
def _norm(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")


QC_LOG = []


def normalize_choice(raw, choices, item_name="", agent_id=None, condition=None, phase=None):
    raw_str = str(raw).strip()
    norm_raw = _norm(raw_str)

    for c in choices:
        if _norm(c) == norm_raw:
            return c, True

    contained = [c for c in choices if _norm(c) in norm_raw]
    if len(contained) == 1:
        QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                            item=item_name, raw=raw_str, matched=contained[0], method="contains"))
        return contained[0], True

    best = difflib.get_close_matches(norm_raw, [_norm(c) for c in choices], n=1, cutoff=0.6)
    if best:
        idx = [_norm(c) for c in choices].index(best[0])
        matched = choices[idx]
        QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                            item=item_name, raw=raw_str, matched=matched, method="fuzzy"))
        return matched, True

    QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                        item=item_name, raw=raw_str, matched=None, method="FALLBACK"))
    return None, False


def normalize_rating(raw, item_name="", agent_id=None, condition=None, phase=None, lo=1, hi=10):
    raw_str = str(raw).strip()
    m = re.search(r"-?\d+", raw_str)
    if not m:
        QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                            item=item_name, raw=raw_str, matched=None, method="FALLBACK"))
        return None, False
    val = int(m.group())
    if not (lo <= val <= hi):
        QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                            item=item_name, raw=raw_str, matched=val, method="out_of_range"))
        return val, False
    if raw_str != str(val):
        QC_LOG.append(dict(agent_id=agent_id, condition=condition, phase=phase,
                            item=item_name, raw=raw_str, matched=val, method="extracted"))
    return val, True


def qc_report():
    if not QC_LOG:
        print("QC: nessuna anomalia registrata (tutte le risposte erano nel formato richiesto).")
        return
    n_fallback = sum(1 for e in QC_LOG if e["method"] == "FALLBACK")
    print(f"QC: {len(QC_LOG)} anomalie totali registrate, di cui {n_fallback} fallback (nessun match).")
    for e in QC_LOG:
        method = e["method"]
        print(f"  [{method:<11}] agente {e['agent_id']} | {e['condition']}/{e['phase']} | "
              f"{e['item']}: raw={e['raw']!r} -> {e['matched']!r}")

print("OK QC/normalizzazione")


OK QC/normalizzazione


## 5 · Stimoli (6 condizioni)

Stessi testi di v1/v2, per piena comparabilita'.


In [5]:
MSG_POS = (
    "Il Comune di Montelago comunica che i lavori di sostituzione completa "
    "della rete idrica nel quartiere inizieranno lunedì prossimo. Il progetto, "
    "finanziato con fondi europei, prevede la posa di nuove tubature su tutti "
    "i tratti principali e si concluderà entro 60 giorni. La qualità dell'acqua "
    "sarà monitorata quotidianamente da tecnici certificati e i risultati saranno "
    "pubblicati ogni settimana sul sito del Comune. Eventuali interruzioni del "
    "servizio saranno comunicate con almeno 48 ore di anticipo. Il Comune "
    "garantisce un numero verde attivo h24 per segnalazioni e informazioni."
)

MSG_NEG = (
    "Il Comune di Montelago comunica che, a causa di ripetuti guasti alle "
    "tubature della rete idrica, si verificheranno interruzioni del servizio "
    "idrico nel quartiere per almeno i prossimi tre mesi. I lavori di riparazione "
    "non possono essere avviati prima dell'autunno per mancanza di fondi. Nelle "
    "ultime analisi, i valori di torbidità dell'acqua hanno superato i limiti "
    "consigliati in alcune vie della zona. Il Comune si scusa per i disagi."
)

MSG_POSW = (
    "Il Comune di Montelago informa che sono in corso valutazioni tecniche per "
    "un possibile intervento di miglioramento della rete idrica nel quartiere. "
    "Non sono ancora disponibili tempistiche definite né dettagli sulle modalità "
    "di intervento. L'amministrazione comunale seguirà l'evoluzione della "
    "situazione e comunicherà eventuali aggiornamenti."
)

MSG_PLA = (
    "Il Comune di Montelago comunica che sabato prossimo si terrà in Piazza "
    "Centrale la Festa del Quartiere, con mercatino dell'artigianato locale, "
    "musica dal vivo e attività per bambini. L'evento è organizzato in "
    "collaborazione con le associazioni di volontariato locali e avrà inizio "
    "alle ore 10:00. L'ingresso è libero e gratuito per tutti i cittadini."
)

CONDITIONS = {
    "POS":  MSG_POS,
    "POS2": MSG_POS,   # replica identica per noise floor (C3)
    "POSW": MSG_POSW,
    "NEG":  MSG_NEG,
    "PLA":  MSG_PLA,
    "CTRL": None,
}
print("OK 6 condizioni:", list(CONDITIONS.keys()))

OK 6 condizioni: ['POS', 'POS2', 'POSW', 'NEG', 'PLA', 'CTRL']


## 6 · Strato di registrazione (storage)

Tre tabelle in formato lungo, popolate automaticamente da ogni chiamata
e da ogni risposta -- pensate per essere scaricate direttamente in
pandas/CSV per l'analisi, senza dover riparsare strutture nidificate:

- **CALL_LOG**: una riga per *ogni* chiamata LLM (qualunque sia il
  motivo) -- timestamp, latenza, token in/out.
- **SURVEY_ROWS**: una riga per (agente, condizione, fase, item) --
  i dati quantitativi del sondaggio, pronti per C1-C7.
- **REACTION_ROWS**: una riga per (agente, condizione, passo) -- il
  dato qualitativo nuovo (la reazione libera della persona).


In [6]:
CALL_LOG = []
SURVEY_ROWS = []
REACTION_ROWS = []


async def call_llm(messages, model=None, max_tokens=400, temperature=None,
                    call_type="generic", agent_id=None, gruppo=None, condition=None):
    model = model or CURRENT_MODEL
    temperature = CURRENT_TEMPERATURE if temperature is None else temperature
    t0 = time.time()
    response = await client.chat.completions.create(
        model=model, messages=messages, max_tokens=max_tokens, temperature=temperature,
    )
    latency = time.time() - t0
    usage = getattr(response, "usage", None)
    CALL_LOG.append({
        "timestamp": _dtclass.now().isoformat(),
        "agent_id": agent_id, "gruppo": gruppo, "condition": condition,
        "call_type": call_type, "model": model, "temperature": temperature,
        "latency_s": round(latency, 3),
        "tokens_in": getattr(usage, "prompt_tokens", None) if usage else None,
        "tokens_out": getattr(usage, "completion_tokens", None) if usage else None,
    })
    return response.choices[0].message.content


def record_survey_response(agent_id, gruppo, condition, model, phase, item_name, value, raw, qc_match):
    SURVEY_ROWS.append({
        "timestamp": _dtclass.now().isoformat(),
        "agent_id": agent_id, "gruppo": gruppo, "condition": condition, "model": model,
        "phase": phase, "item": item_name, "value": value, "raw": raw, "qc_match": qc_match,
    })


def record_reaction(agent_id, gruppo, condition, model, step, marker, reaction):
    REACTION_ROWS.append({
        "timestamp": _dtclass.now().isoformat(),
        "agent_id": agent_id, "gruppo": gruppo, "condition": condition, "model": model,
        "step": step, "marker": marker, "reaction": reaction,
    })

print("OK strato di registrazione (CALL_LOG, SURVEY_ROWS, REACTION_ROWS)")


OK strato di registrazione (CALL_LOG, SURVEY_ROWS, REACTION_ROWS)


## 7 · Battery (side-query, non modifica la narrazione)

Punto architetturale chiave: la batteria prende la `narrative` cosi'
com'e' in quel momento, ci aggiunge la singola domanda dell'item, manda
tutto al modello, **non** rimette la risposta nella narrazione. Il
sondaggio non si mescola mai con cio' che "succede" alla persona --
ed e' per questo che il costo e' fisso (5 o 6 chiamate), mai variabile.


In [7]:
async def ask_battery_minimal(narrative, phase, agent_id, gruppo, condition, model=None):
    model = model or CURRENT_MODEL
    items = list(ITEMS)
    if INCLUDE_OPEN_ENDED and phase == "POST":
        items = items + [OPEN_ENDED_ITEM]

    out = {}
    for item in items:
        prompt_text = item["prompt"]
        if item["response_type"] == "choice":
            prompt_text += " Opzioni: " + ", ".join(item["choices"]) + "."
        query = narrative + [{"role": "user", "content": prompt_text}]
        raw = await call_llm(query, model=model, max_tokens=60, call_type="survey_item",
                              agent_id=agent_id, gruppo=gruppo, condition=condition)

        if item["response_type"] == "integer":
            val, ok = normalize_rating(raw, item_name=item["name"], agent_id=agent_id,
                                        condition=condition, phase=phase)
        elif item["response_type"] == "choice":
            val, ok = normalize_choice(raw, item["choices"], item_name=item["name"],
                                        agent_id=agent_id, condition=condition, phase=phase)
        else:  # open_text
            val, ok = raw, True

        record_survey_response(agent_id, gruppo, condition, model, phase,
                                item["name"], val, raw, ok)
        out[item["name"]] = val
    return out

print("OK ask_battery_minimal")


OK ask_battery_minimal


## 8 · Esecuzione di una persona sotto una condizione

PRE -> [iniezione stimolo -> 3 reazioni libere, con marcatori temporali,
salvate sia in `narrative` che in `REACTION_ROWS`] -> POST. Nessun ciclo
di tool-decision: ogni "passo" e' esattamente una chiamata.

Il modello e' lasciato libero di reagire come vuole (nessun prompt
guidato passo-per-passo) -- piu' fedele allo spirito di SIVE, al costo
di non poter controllare quanto la reazione resti "in tema".


In [ ]:
TIME_MARKERS = [
    None,  # passo 0: reazione immediata allo stimolo
    "[Sono passate un paio d'ore. Hai continuato a pensarci.]",
    "[E' passato il resto della giornata.]",
]


async def run_persona_minimal(record, condition_label, message, model=None,
                                n_reaction_turns=3):
    model = model or CURRENT_MODEL
    agent_id = record["id"]
    gruppo = record["gruppo"]
    profile = build_profile(record)
    narrative = [{"role": "system", "content": build_system_prompt(profile)}]

    pre = await ask_battery_minimal(narrative, "PRE", agent_id, gruppo, condition_label, model)

    reazioni = []
    if message is not None:
        for i in range(n_reaction_turns):
            prompt = (f"[Comunicazione del Comune di Montelago]\n{message}"
                      if i == 0 else TIME_MARKERS[i])
            narrative.append({"role": "user", "content": prompt})
            reazione = await call_llm(narrative, model=model, max_tokens=400,
                                       call_type="reaction", agent_id=agent_id,
                                       gruppo=gruppo, condition=condition_label)
            narrative.append({"role": "assistant", "content": reazione})
            record_reaction(agent_id, gruppo, condition_label, model, i, prompt, reazione)
            reazioni.append({"step": i, "marker": prompt, "reazione": reazione})

    post = await ask_battery_minimal(narrative, "POST", agent_id, gruppo, condition_label, model)

    return {"pre": pre, "post": post, "reazioni": reazioni, "narrative": narrative}

print("OK run_persona_minimal")


OK run_persona_minimal


## 9 · Salvataggio incrementale su Drive

Lezione del 18 giugno 2026 (smoke test da 2 ore perso per un bug,
nessun checkpoint intermedio): salviamo dopo *ogni condizione completata*,
non solo a fine campagna. Tre file CSV (pronti per pandas/R, niente da
riparsare) piu' un bundle JSON con i metadati.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/SyntheticPopulation/SIVE/results_minimal2')
RUN_TIMESTAMP = _dtclass.now().strftime("%Y%m%d_%H%M%S")

# Subfolder dedicato a questo run: modello + temperatura + timestamp
# (cosi' ogni run e' un'unita' autonoma, facile da archiviare/confrontare)
_safe_model = CURRENT_MODEL.replace("/", "_")
_temp_tag = f"t{CURRENT_TEMPERATURE}".replace(".", "")
RUN_DIR = BASE_DIR / f"{_safe_model}_{_temp_tag}_{RUN_TIMESTAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Output di questo run in:", RUN_DIR)


def save_checkpoint(label=""):
    """Salva i tre CSV + meta nel subfolder del run corrente.
    I nomi dei file sono semplici (survey_POS.csv, non il tag lungo
    ripetuto) perche' il contesto e' gia' dato dalla cartella."""
    suffix = f"_{label}" if label else ""

    pd.DataFrame(SURVEY_ROWS).to_csv(RUN_DIR / f"survey{suffix}.csv", index=False)
    pd.DataFrame(REACTION_ROWS).to_csv(RUN_DIR / f"reactions{suffix}.csv", index=False)
    pd.DataFrame(CALL_LOG).to_csv(RUN_DIR / f"calls{suffix}.csv", index=False)

    meta = {
        "model": CURRENT_MODEL, "temperature": CURRENT_TEMPERATURE,
        "timestamp": RUN_TIMESTAMP, "checkpoint_label": label,
        "run_dir": str(RUN_DIR),
        "n_survey_rows": len(SURVEY_ROWS), "n_reaction_rows": len(REACTION_ROWS),
        "n_calls": len(CALL_LOG),
        "conditions_so_far": sorted(set(r["condition"] for r in SURVEY_ROWS)),
    }
    with open(RUN_DIR / f"meta{suffix}.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"Checkpoint [{label or 'anon'}]: {meta['n_survey_rows']} righe survey, "
          f"{meta['n_reaction_rows']} reazioni, {meta['n_calls']} chiamate")

print("OK save_checkpoint")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output di questo run in: /content/drive/MyDrive/Colab Notebooks/SyntheticPopulation/SIVE/results_minimal2/deepseek_deepseek-chat_t09_20260629_202846
OK save_checkpoint


## 10 · Esecuzione della campagna completa (120 agenti × 7 condizioni)

Esecuzione su tutti gli agenti (`all_ids`). Le righe per un eventuale
pilot a 6 agenti (2 per gruppo LOW/MED/HIGH) restano disponibili più
sotto, commentate, per chi voglia rieseguire un test ridotto prima di
una campagna completa su un nuovo modello o endpoint.


In [ ]:
# Per un pilot ridotto (2 agenti per gruppo, utile prima di una campagna
# piena su un nuovo modello/endpoint), decommentare la riga seguente:
# all_ids = GRUPPI["LOW"][:2] + GRUPPI["MED"][:2] + GRUPPI["HIGH"][:2]
all_ids = GRUPPI["LOW"] + GRUPPI["MED"] + GRUPPI["HIGH"]

for condition_label, message in CONDITIONS.items():
    print(f"\n=== CONDIZIONE [{condition_label}] ===")
    for aid in all_ids:
        record = META[aid]
        res = await run_persona_minimal(record, condition_label, message)
        print(f"  agente {aid} ({record['gruppo']}, {record['persona']})")
        print("    PRE :", res["pre"])
        print("    POST:", res["post"])
    save_checkpoint(label=condition_label)



=== CONDIZIONE [POS] ===
  agente 1 (LOW, sfiduciato critico)
    PRE : {'fiducia_istituzione': 2, 'credibilita': 3, 'adeguatezza_info': 2, 'emozione': 'rabbia', 'intenzione': 'non cambierà nulla per me'}
    POST: {'fiducia_istituzione': 2, 'credibilita': 3, 'adeguatezza_info': 2, 'emozione': 'rabbia', 'intenzione': 'non cambierà nulla per me', 'motivazione_libera': 'Perché dopo anni di promesse non mantenute e lavori fatti male, ho perso ogni fiducia nel Comune. So già che questi lavori ci lasceranno solo disagi e nessun reale miglioramento.'}
  agente 2 (LOW, delusa amareggiata)
    PRE : {'fiducia_istituzione': 3, 'credibilita': 3, 'adeguatezza_info': 2, 'emozione': 'rabbia', 'intenzione': 'contattare il Comune per avere chiarimenti'}
    POST: {'fiducia_istituzione': 2, 'credibilita': 5, 'adeguatezza_info': 3, 'emozione': 'rabbia', 'intenzione': 'parlare con i vicini della situazione', 'motivazione_libera': "Perché dopo anni di promesse non mantenute e servizi scadenti, ho impara

## 11 · QC report

In [ ]:
qc_report()


QC: nessuna anomalia registrata (tutte le risposte erano nel formato richiesto).


## 12 · Tempi (dal CALL_LOG)


In [ ]:
df_calls = pd.DataFrame(CALL_LOG)
print(f"Chiamate totali: {len(df_calls)}")
print(f"Latenza media: {df_calls['latency_s'].mean():.2f}s "
      f"(min {df_calls['latency_s'].min():.2f}s, max {df_calls['latency_s'].max():.2f}s)")
print("\nPer tipo di chiamata:")
print(df_calls.groupby("call_type")[["latency_s"]].agg(["count", "mean", "sum"]))


## 13 · Salvataggio finale completo


In [ ]:
save_checkpoint(label="FINAL")
print("Tutti i file di questo run sono in:", RUN_DIR)


Checkpoint [FINAL]: 0 righe survey, 0 reazioni, 0 chiamate, costo finora $0.0000
Tutti i file di questo run sono in: /content/drive/MyDrive/Colab Notebooks/SyntheticPopulation/SIVE/results_minimal2/deepseek_deepseek-chat_t07_20260626_091423


## 14 · Run incrementale: aggiunta della condizione POSW2

La condizione POSW, pensata come segnale debolmente positivo, si è
rivelata nell'analisi funzionalmente negativa (vedi paper, sezione sul
sistema sperimentale Montelago). POSW2 è la versione ricalibrata,
somministrata in un secondo momento agli stessi 120 agenti e accodata
ai dati del run originale, senza rieseguire le altre sei condizioni.

**Prerequisiti**: le celle 1-9 di questo notebook devono essere già
state eseguite in questa sessione (client, `PROFILI`, `META`, `GRUPPI`,
funzioni, `RUN_DIR`), e i CSV del run originale devono già esistere in
`RUN_DIR`.


In [ ]:
import asyncio

_already_done = set(
    r["condition"] for r in SURVEY_ROWS if r["condition"] == "POSW2"
)
assert not _already_done, "POSW2 già nei dati — non rieseguire."

all_ids = GRUPPI["LOW"] + GRUPPI["MED"] + GRUPPI["HIGH"]

print("=== CONDIZIONE [POSW2] — run incrementale ===")
for aid in all_ids:
    record = META[aid]
    res = await run_persona_minimal(record, "POSW2", MSG_POSW2)
    print(f"  agente {aid} ({record['gruppo']}, {record['persona'][:30]})")
    print(f"    PRE : {res['pre']}")
    print(f"    POST: {res['post']}")

# Checkpoint dedicato POSW2 (non sovrascrive FINAL, è additivo)
save_checkpoint(label="POSW2")
print("\n✅  Run POSW2 completato e salvato.")

# --- Analisi rapida: confronto POSW vs POSW2 ---
print("\n=== Confronto rapido POSW vs POSW2 (fiducia_istituzione) ===")

def _delta(cond, col="fiducia_istituzione"):
    df = pd.DataFrame(SURVEY_ROWS)
    pre  = df[(df.condition == cond) & (df.phase == "PRE")  & (df.item == col)]\
             .set_index("agent_id")["value"].astype(float)
    post = df[(df.condition == cond) & (df.phase == "POST") & (df.item == col)]\
             .set_index("agent_id")["value"].astype(float)
    return (post - pre).dropna()

d_posw  = _delta("POSW")
d_posw2 = _delta("POSW2")

print(f"  POSW  delta medio: {d_posw.mean():+.3f}  (sd={d_posw.std():.3f})")
print(f"  POSW2 delta medio: {d_posw2.mean():+.3f}  (sd={d_posw2.std():.3f})")

# t-test POSW2 vs CTRL (atteso: POSW2 > CTRL, cioè delta > 0 o almeno > CTRL)
d_ctrl  = _delta("CTRL")
from scipy import stats as _stats
t_vs_ctrl, p_vs_ctrl = _stats.ttest_ind(d_posw2, d_ctrl)
print(f"\n  POSW2 vs CTRL: t={t_vs_ctrl:.3f}, p={p_vs_ctrl:.4f}")
print(f"  (atteso: POSW2 > CTRL, p < 0.05 se calibrazione ok)")

# Ordinamento C5 aggiornato
for cond in ["NEG", "CTRL", "PLA", "POSW", "POSW2", "POS2", "POS"]:
    d = _delta(cond)
    print(f"  {cond:<6} delta medio: {d.mean():+.3f}")

# Salvataggio finale aggiornato (include tutto: originale + POSW2)
save_checkpoint(label="FINAL_v2")
print("\n✅  Salvataggio finale aggiornato: FINAL_v2")
print(f"   Tutti i file in: {RUN_DIR}")

=== CONDIZIONE [POSW2] — run incrementale ===
  agente 1 (LOW, sfiduciato critico)
    PRE : {'fiducia_istituzione': 1, 'credibilita': 2, 'adeguatezza_info': 1, 'emozione': 'rabbia', 'intenzione': 'contattare il Comune per avere chiarimenti'}
    POST: {'fiducia_istituzione': 2, 'credibilita': 2, 'adeguatezza_info': 1, 'emozione': 'rabbia', 'intenzione': 'non cambierà nulla per me', 'motivazione_libera': 'Perché dopo anni di promesse e "controlli" le tubature sono ancora quelle di quando mi sono trasferito qui, e ogni volta che c\'è un problema dobbiamo aspettare mesi prima che intervengano!  \n\nE la cosa che mi fa più incazz'}
  agente 2 (LOW, delusa amareggiata)
    PRE : {'fiducia_istituzione': 1, 'credibilita': 2, 'adeguatezza_info': 1, 'emozione': 'rabbia', 'intenzione': 'contattare il Comune per avere chiarimenti'}
    POST: {'fiducia_istituzione': 1, 'credibilita': 4, 'adeguatezza_info': 2, 'emozione': 'rabbia', 'intenzione': 'contattare il Comune per avere chiarimenti', 'motiv